# Demo negFIN — Google Colab
    
Mở trong Google Colab và chọn **Runtime → Run all**. Notebook lấy source công khai theo `REF` và chuẩn bị Julia nếu cần.

Notebook chạy ví dụ toy, Online Retail (giỏ hàng) và Splice-junction (pattern vị trí đã cho theo lớp). Mọi frequent itemset do CLI negFIN của nhóm sinh ra. Dữ liệu UCI được kiểm SHA-256 trước khi dùng.

In [1]:
# Tham số — đổi REF thành tên nhánh hoặc commit để chạy thử một PR.
REPO_URL = "https://github.com/DankDaPancake/CSC14004-group4-negfin.git"
REF = "main"
VARIANT = "baseline"          # hoặc "optimized"
RETAIL_MINSUP = "0.01"        # chuỗi thập phân theo docs/baseline-contract.md
RETAIL_MINCONF = 0.5
SPLICE_MINSUP = "0.3"

JULIA_VERSION = "1.12.0"
JULIA_URL = f"https://julialang-s3.julialang.org/bin/linux/x64/1.12/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
JULIA_SHA256 = "6f87b8fcf5ef6a7371e8c79d948aedfa0ba28ce44447c446d7d82e70f0158da8"

## 1. Môi trường: mã nguồn, Julia, thư viện Python

In [2]:
import hashlib, os, platform, shutil, subprocess, sys, tarfile, tomllib, urllib.request, zipfile
from pathlib import Path

def sh(*cmd, cwd=None, tail=6000):
    print("$", " ".join(map(str, cmd)))
    r = subprocess.run(list(map(str, cmd)), cwd=cwd, env={**os.environ, "PYTHONUTF8": "1"}, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout[-tail:], r.stderr[-tail:], sep="")
    r.check_returncode()
    return r.stdout

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download(url, destination, expected_sha256):
    """Download with a network timeout and verify bytes before publishing the file."""
    part = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "negFIN-course-demo/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response, part.open("wb") as output:
            shutil.copyfileobj(response, output, length=1 << 20)
        if sha256(part) != expected_sha256:
            raise ValueError(f"SHA-256 không khớp: {url}")
        part.replace(destination)
    finally:
        part.unlink(missing_ok=True)

ROOT = Path("/content/negfin")
if not ROOT.exists():
    sh("git", "clone", "--quiet", REPO_URL, ROOT)
sh("git", "fetch", "--quiet", "origin", REF, cwd=ROOT)
sh("git", "checkout", "--quiet", "--detach", "FETCH_HEAD", cwd=ROOT)
os.chdir(ROOT)
COMMIT = sh("git", "rev-parse", "HEAD").strip()

$ git clone --quiet https://github.com/DankDaPancake/CSC14004-group4-negfin.git /content/negfin

$ git fetch --quiet origin d75810cbf1132863979432502882cc7730bcecad

$ git checkout --quiet --detach FETCH_HEAD

$ git rev-parse HEAD
d75810cbf1132863979432502882cc7730bcecad



In [3]:
JULIA = shutil.which("julia")
if JULIA is None:
    assert platform.system() == "Linux", "Không tìm thấy julia: hãy cài Julia >= 1.9 rồi chạy lại."
    tgz = Path("/tmp/julia.tar.gz")
    download(JULIA_URL, tgz, JULIA_SHA256)
    with tarfile.open(tgz) as t:
        t.extractall("/opt", filter="data")
    JULIA = f"/opt/julia-{JULIA_VERSION}/bin/julia"
sh(JULIA, "--version")

from importlib.util import find_spec
missing = [p for p in ("pandas", "openpyxl") if find_spec(p) is None]
if missing:
    sh(sys.executable, "-m", "pip", "install", "--quiet", *missing)
import pandas as pd
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| commit", COMMIT)

def negfin(inp, out, minsup):
    """Gọi CLI negFIN của nhóm; trả về stats TOML."""
    stats = Path(str(out).removesuffix(".txt") + ".stats.toml")
    sh(JULIA, "--startup-file=no", "--project=.", "run.jl", "--input", inp, "--output", out,
       "--minsup", minsup, "--variant", VARIANT, "--stats", stats)
    with open(stats, "rb") as f:
        return tomllib.load(f)

$ /usr/local/bin/julia --version
julia version 1.12.6

Python 3.13.18 | pandas 2.2.3 | commit d75810cbf1132863979432502882cc7730bcecad


**Thư viện Python.** `openpyxl` giúp đọc workbook Excel của Online Retail. `pandas` đọc bảng, chuẩn bị dữ liệu Retail và hiển thị các luật kết hợp. Các bước khai thác itemset chạy bằng Julia qua `run.jl`.

## 2. Dữ liệu công khai

Nguồn, DOI, giấy phép (CC BY 4.0) và SHA-256 nằm trong `data/applications/manifest.toml`. File đã có và đúng checksum thì không tải lại.

In [4]:
RAW = Path("data/applications/raw")
with open("data/applications/manifest.toml", "rb") as f:
    manifest = tomllib.load(f)["datasets"]

for d in manifest:
    target = RAW / d["path"]
    if not (target.exists() and sha256(target) == d["sha256"]):
        folder = RAW / d["name"]
        folder.mkdir(parents=True, exist_ok=True)
        z = folder / "download.zip"
        download(d["source_url"], z, d["zip_sha256"])
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
    assert sha256(target) == d["sha256"], f"SHA-256 {target} không khớp"
    print(f"OK {d['title']}: {target} ({d['license']}, doi:{d['doi']})")

OK Online Retail: data/applications/raw/retail/Online Retail.xlsx (CC BY 4.0, doi:10.24432/C5BW33)
OK Molecular Biology (Splice-junction Gene Sequences): data/applications/raw/splice/splice.data (CC BY 4.0, doi:10.24432/C5M888)


## 3. Kiểm tra nhanh CLI trên ví dụ toy

In [5]:
OUT = Path("results/demo")
OUT.mkdir(parents=True, exist_ok=True)
negfin("data/toy/example.txt", OUT / "toy.txt", "0.4")
def itemsets(path):
    return {line.strip() for line in open(path, encoding="utf-8") if line.strip()}
assert itemsets(OUT / "toy.txt") == itemsets("data/toy/example.expected.txt")
print("toy: khớp", len(itemsets(OUT / "toy.txt")), "itemset với kết quả mong đợi")

$ /usr/local/bin/julia --startup-file=no --project=. run.jl --input data/toy/example.txt --output results/demo/toy.txt --minsup 0.4 --variant baseline --stats results/demo/toy.stats.toml

toy: khớp 15 itemset với kết quả mong đợi


## 4. Ứng dụng 1 — Giỏ hàng (Online Retail)

Một hóa đơn là một transaction, item là `StockCode` sản phẩm. Bỏ hóa đơn hủy/điều chỉnh, `Quantity ≤ 0`, `UnitPrice ≤ 0` và mã không phải sản phẩm (xem `docs/applications-proposal.md`).
Luật `X ⇒ Y`: support = sup(X∪Y)/N, confidence = sup(X∪Y)/sup(X), lift = confidence/(sup(Y)/N).

In [6]:
R = OUT / "retail"
sh(sys.executable, "apps/retail/prepare.py", "--input", RAW / "retail/Online Retail.xlsx", "--outdir", R)
retail_stats = negfin(R / "transactions.txt", R / f"itemsets_{RETAIL_MINSUP}.txt", RETAIL_MINSUP)
print({k: retail_stats[k] for k in ("n_transactions", "min_count", "n_patterns", "total_seconds")})

$ /usr/bin/python3 apps/retail/prepare.py --input data/applications/raw/retail/Online Retail.xlsx --outdir results/demo/retail
raw	rows=541909	invoices=25900
bo_hoa_don_huy_dieu_chinh (C*, A*)	rows=532618	invoices=22061
bo_quantity_<=0	rows=531282	invoices=20725
bo_unitprice_<=0	rows=530103	invoices=19959
chi_giu_ma_san_pham	rows=527725	invoices=19773
transactions=19773
items=3791
avg_items_per_transaction=26.16

$ /usr/local/bin/julia --startup-file=no --project=. run.jl --input results/demo/retail/transactions.txt --output results/demo/retail/itemsets_0.01.txt --minsup 0.01 --variant baseline --stats results/demo/retail/itemsets_0.01.stats.toml

{'n_transactions': 19773, 'min_count': 198, 'n_patterns': 2026, 'total_seconds': 21.384584882}


In [7]:
sh(sys.executable, "apps/retail/rules.py", "--itemsets", R / f"itemsets_{RETAIL_MINSUP}.txt",
   "--stats", R / f"itemsets_{RETAIL_MINSUP}.stats.toml", "--mapping", R / "mapping.csv",
   "--output", R / "rules.csv", "--minconf", RETAIL_MINCONF)
rules = pd.read_csv(R / "rules.csv")
rules.head(10)[["antecedent_names", "consequent_names", "support_count", "support", "confidence", "lift"]]

$ /usr/bin/python3 apps/retail/rules.py --itemsets results/demo/retail/itemsets_0.01.txt --stats results/demo/retail/itemsets_0.01.stats.toml --mapping results/demo/retail/mapping.csv --output results/demo/retail/rules.csv --minconf 0.5
N=19773 itemsets=2026 minconf=0.5 rules=962

# top theo lift
lift= 78.77 conf=0.86 sup=0.0103 (203)  HERB MARKER THYME  =>  HERB MARKER ROSEMARY + HERB MARKER PARSLEY
lift= 78.77 conf=0.94 sup=0.0103 (203)  HERB MARKER ROSEMARY + HERB MARKER PARSLEY  =>  HERB MARKER THYME
lift= 78.52 conf=0.95 sup=0.0103 (203)  HERB MARKER THYME + HERB MARKER PARSLEY  =>  HERB MARKER ROSEMARY
lift= 78.52 conf=0.85 sup=0.0103 (203)  HERB MARKER ROSEMARY  =>  HERB MARKER THYME + HERB MARKER PARSLEY
lift= 78.46 conf=0.95 sup=0.0101 (200)  HERB MARKER THYME + HERB MARKER BASIL  =>  HERB MARKER ROSEMARY
lift= 78.46 conf=0.83 sup=0.0101 (200)  HERB MARKER ROSEMARY  =>  HERB MARKER THYME + HERB MARKER BASIL
lift= 77.97 conf=0.93 sup=0.0101 (200)  HERB MARKER ROSEMARY + HERB MA

,antecedent_names,consequent_names,support_count,support,confidence,lift
0,HERB MARKER THYME,HERB MARKER ROSEMARY + HERB MARKER PARSLEY,203,0.010267,0.856540,78.773800
1,HERB MARKER ROSEMARY + HERB MARKER PARSLEY,HERB MARKER THYME,203,0.010267,0.944186,78.773800
2,HERB MARKER THYME + HERB MARKER PARSLEY,HERB MARKER ROSEMARY,203,0.010267,0.953052,78.519542
3,HERB MARKER ROSEMARY,HERB MARKER THYME + HERB MARKER PARSLEY,203,0.010267,0.845833,78.519542
4,HERB MARKER THYME + HERB MARKER BASIL,HERB MARKER ROSEMARY,200,0.010115,0.952381,78.464286
5,HERB MARKER ROSEMARY,HERB MARKER THYME + HERB MARKER BASIL,200,0.010115,0.833333,78.464286
6,HERB MARKER ROSEMARY + HERB MARKER BASIL,HERB MARKER THYME,200,0.010115,0.934579,77.972318
7,HERB MARKER THYME,HERB MARKER ROSEMARY + HERB MARKER BASIL,200,0.010115,0.843882,77.972318
8,HERB MARKER THYME,HERB MARKER ROSEMARY,221,0.011177,0.932489,76.825475
9,HERB MARKER ROSEMARY,HERB MARKER THYME,221,0.011177,0.920833,76.825475


## 5. Ứng dụng 2 — Sinh học (Splice-junction)

Một chuỗi DNA 60 base là một transaction; item là `p<vị trí>_<base>` (−30..−1, +1..+30). negFIN chạy riêng từng lớp: EI (donor), IE (acceptor), N (không phải điểm nối).
`growth` = support trong lớp / support trong các lớp còn lại.

In [8]:
S = OUT / "splice"
sh(sys.executable, "apps/splice/prepare.py", "--input", RAW / "splice/splice.data", "--outdir", S)
splice_stats = {}
for c in ("EI", "IE", "N"):
    splice_stats[c] = negfin(S / f"transactions_{c}.txt", S / f"itemsets_{c}_{SPLICE_MINSUP}.txt", SPLICE_MINSUP)
    print(c, {k: splice_stats[c][k] for k in ("n_transactions", "min_count", "n_patterns")})

$ /usr/bin/python3 apps/splice/prepare.py --input data/applications/raw/splice/splice.data --outdir results/demo/splice
sequences=3190
class_EI=767
class_IE=768
class_N=1655
sequences_with_ambiguous_bases=15

$ /usr/local/bin/julia --startup-file=no --project=. run.jl --input results/demo/splice/transactions_EI.txt --output results/demo/splice/itemsets_EI_0.3.txt --minsup 0.3 --variant baseline --stats results/demo/splice/itemsets_EI_0.3.stats.toml

EI {'n_transactions': 767, 'min_count': 231, 'n_patterns': 280}
$ /usr/local/bin/julia --startup-file=no --project=. run.jl --input results/demo/splice/transactions_IE.txt --output results/demo/splice/itemsets_IE_0.3.txt --minsup 0.3 --variant baseline --stats results/demo/splice/itemsets_IE_0.3.stats.toml

IE {'n_transactions': 768, 'min_count': 231, 'n_patterns': 331}
$ /usr/local/bin/julia --startup-file=no --project=. run.jl --input results/demo/splice/transactions_N.txt --output results/demo/splice/itemsets_N_0.3.txt --minsup 0.3 --var

In [9]:
for c in ("EI", "IE"):
    sh(sys.executable, "apps/splice/compare.py", "--target", c, "--itemsets", S / f"itemsets_{c}_{SPLICE_MINSUP}.txt",
       "--data", S, "--output", S / f"compare_{c}.csv", "--top", "8")

$ /usr/bin/python3 apps/splice/compare.py --target EI --itemsets results/demo/splice/itemsets_EI_0.3.txt --data results/demo/splice --output results/demo/splice/compare_EI.csv --top 8
target=EI itemsets=280 N_EI=767 N_IE=768 N_N=1655
growth=    inf  EI=30.2% IE=0.0% N=0.0%  p+1_G + p+2_T + p+4_A + p+5_G + p+6_T
growth= 488.07  EI=40.3% IE=0.1% N=0.1%  p+1_G + p+2_T + p+5_G + p+6_T
growth= 263.26  EI=32.6% IE=0.1% N=0.1%  p+1_G + p+2_T + p+3_A + p+4_A
growth= 244.30  EI=30.2% IE=0.1% N=0.1%  p+2_T + p+4_A + p+5_G + p+6_T
growth= 193.10  EI=63.8% IE=0.9% N=0.1%  p+1_G + p+2_T + p+4_A + p+5_G
growth= 169.96  EI=35.1% IE=0.7% N=0.0%  p+1_G + p+2_T + p+3_G + p+4_A + p+5_G
growth= 163.82  EI=47.3% IE=0.9% N=0.0%  p-1_G + p+1_G + p+2_T + p+4_A + p+5_G
growth= 151.00  EI=31.2% IE=0.4% N=0.1%  p-25_G + p+1_G + p+2_T + p+5_G

$ /usr/bin/python3 apps/splice/compare.py --target IE --itemsets results/demo/splice/itemsets_IE_0.3.txt --data results/demo/splice --output results/demo/splice/compare_IE.

Kỳ vọng (consensus đã công bố): donor `…AG|GTRAGT…` ở EI và acceptor `…(Y)n CAG|G…` ở IE; lớp N không có pattern ở minsup 0,3. Diễn giải đầy đủ: `docs/applications-splice-results.md`, `docs/applications-retail-results.md`.

## 6. Thông tin tái lập

In [10]:
print("commit  :", COMMIT)
print("julia   :", sh(JULIA, "--version").strip())
print("variant :", VARIANT)
print("retail  :", f"N={retail_stats['n_transactions']} minsup={RETAIL_MINSUP} itemsets={retail_stats['n_patterns']} minconf={RETAIL_MINCONF} rules={len(rules)}")
print("splice  :", f"minsup={SPLICE_MINSUP} " + " ".join(f"{c}:{s['n_transactions']}->{s['n_patterns']}" for c, s in splice_stats.items()))

if (RETAIL_MINSUP, RETAIL_MINCONF) == ("0.01", 0.5):
    assert (retail_stats["n_transactions"], retail_stats["n_patterns"], len(rules)) == (19773, 2026, 962)
if SPLICE_MINSUP == "0.3":
    assert {name: (s["n_transactions"], s["n_patterns"]) for name, s in splice_stats.items()} == {
        "EI": (767, 280), "IE": (768, 331), "N": (1655, 0)
    }

commit  : d75810cbf1132863979432502882cc7730bcecad
$ /usr/local/bin/julia --version
julia version 1.12.6

julia   : julia version 1.12.6
variant : baseline
retail  : N=19773 minsup=0.01 itemsets=2026 minconf=0.5 rules=962
splice  : minsup=0.3 EI:767->280 IE:768->331 N:1655->0
